# Small Llama-style LLM — full walkthrough

A self-contained build of the whole model, one piece per section: device, tokenizer, embedding, RMSNorm,
RoPE, attention, MLP, transformer block, the full model, causality check and sampling. All model code is
written in the notebook; the only imports are external libraries.

The model has **random weights**, so generated text is gibberish until you train it or load published
weights. The sizes below are the tiny `nano` ones so everything runs in seconds.

Select the kernel **smalllm (.venv)** before running.

In [1]:
import math
from dataclasses import dataclass

import torch
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from tokenizers import Tokenizer
from torch import Tensor, nn

torch.manual_seed(0)

C:\projects\smallllm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Device and dtype
Cuda if available, else Apple MPS, else CPU. bf16 on capable CUDA GPUs, otherwise fp32.

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

dtype = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float32
print(f"device={device} dtype={dtype}")

device=cpu dtype=torch.float32


## 2. Config
The `nano` size. For the published SmolLM2-135M use hidden 576, 30 layers, 9 heads, 3 kv heads,
intermediate 1536, max positions 2048.

In [3]:
@dataclass
class ModelConfig:
    vocab_size: int = 49152
    hidden_size: int = 128
    num_layers: int = 4
    num_heads: int = 2
    num_kv_heads: int = 1
    intermediate_size: int = 352
    head_dim: int = 64
    rope_theta: float = 100000.0
    rms_norm_eps: float = 1e-5
    max_position_embeddings: int = 128
    tie_word_embeddings: bool = True


cfg = ModelConfig()
cfg

ModelConfig(vocab_size=49152, hidden_size=128, num_layers=4, num_heads=2, num_kv_heads=1, intermediate_size=352, head_dim=64, rope_theta=100000.0, rms_norm_eps=1e-05, max_position_embeddings=128, tie_word_embeddings=True)

## 3. Tokenizer
Text to ids and back. Downloads `tokenizer.json` (small) on first run, then uses the cache. `Ġ` marks a leading space.

In [4]:
tok = Tokenizer.from_file(hf_hub_download("HuggingFaceTB/SmolLM2-135M", "tokenizer.json"))

text = "Once upon a time"
ids = tok.encode(text).ids
print(ids)
for i in ids:
    print(f"{i:>6} -> {tok.id_to_token(i)!r}")
print(repr(tok.decode(ids)))

[6403, 1980, 253, 655]
  6403 -> 'Once'
  1980 -> 'Ġupon'
   253 -> 'Ġa'
   655 -> 'Ġtime'
'Once upon a time'


## 4. RMSNorm
Rescales each token vector to unit root-mean-square (computed in float32), then applies a learned per-dimension weight.

In [5]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float) -> None:
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x: Tensor) -> Tensor:
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return self.weight * xf.to(x.dtype)

## 5. Rotary position embeddings (RoPE)
Position is encoded by *rotating* q and k. Each pair of dimensions is rotated by an angle proportional to the
token's position, so attention scores depend only on relative distance.

Layout matters for published weights: the Hugging Face convention pairs the first half of `head_dim` with the
second half (`rotate_half`), not interleaved even/odd pairs.

In [6]:
def rope_cos_sin(positions: Tensor, head_dim: int, theta: float) -> tuple[Tensor, Tensor]:
    """cos/sin tables for integer positions, shape (T, head_dim)."""
    inv_freq = 1.0 / theta ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim)
    freqs = positions.float()[:, None] * inv_freq.to(positions.device)[None, :]
    emb = torch.cat((freqs, freqs), dim=-1)  # dims i and i + head_dim/2 share one angle
    return emb.cos(), emb.sin()


def rotate_half(x: Tensor) -> Tensor:
    x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x: Tensor, cos: Tensor, sin: Tensor) -> Tensor:
    """x: (B, n_heads, T, head_dim); cos/sin: (T, head_dim)."""
    cos, sin = cos.to(x.dtype), sin.to(x.dtype)
    return x * cos + rotate_half(x) * sin

Test the key property: shifting every position by the same offset leaves attention scores unchanged.

In [7]:
T, hd = 6, cfg.head_dim
q = torch.randn(1, 1, T, hd)
k = torch.randn(1, 1, T, hd)


def scores(offset: int) -> Tensor:
    cos, sin = rope_cos_sin(torch.arange(T) + offset, hd, cfg.rope_theta)
    return apply_rope(q, cos, sin) @ apply_rope(k, cos, sin).transpose(-1, -2)


print("shift-invariant:", torch.allclose(scores(0), scores(1000), atol=1e-3))

shift-invariant: True


## 6. Attention
Grouped-query attention: `num_heads` query heads share `num_kv_heads` key/value heads. No biases. RoPE is
applied to q and k only. `is_causal=True` stops each token looking at later tokens.

In [8]:
class Attention(nn.Module):
    def __init__(self, cfg: ModelConfig) -> None:
        super().__init__()
        assert cfg.num_heads % cfg.num_kv_heads == 0
        self.n_heads, self.n_kv_heads, self.head_dim = cfg.num_heads, cfg.num_kv_heads, cfg.head_dim
        h = cfg.hidden_size
        self.q_proj = nn.Linear(h, self.n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(h, self.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(h, self.n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(self.n_heads * self.head_dim, h, bias=False)

    def forward(self, x: Tensor, cos: Tensor, sin: Tensor) -> Tensor:
        b, t, _ = x.shape
        q = self.q_proj(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(b, t, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(b, t, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        n_rep = self.n_heads // self.n_kv_heads  # each kv head serves n_rep consecutive query heads
        k, v = k.repeat_interleave(n_rep, dim=1), v.repeat_interleave(n_rep, dim=1)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o_proj(out.transpose(1, 2).reshape(b, t, -1))

## 7. MLP (SwiGLU)
`down_proj(silu(gate_proj(x)) * up_proj(x))`: the gate decides how much of each hidden feature passes through.

In [9]:
class MLP(nn.Module):
    def __init__(self, cfg: ModelConfig) -> None:
        super().__init__()
        self.gate_proj = nn.Linear(cfg.hidden_size, cfg.intermediate_size, bias=False)
        self.up_proj = nn.Linear(cfg.hidden_size, cfg.intermediate_size, bias=False)
        self.down_proj = nn.Linear(cfg.intermediate_size, cfg.hidden_size, bias=False)

    def forward(self, x: Tensor) -> Tensor:
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

## 8. Transformer block
Pre-norm with residuals: `x + attention(norm(x))`, then `x + mlp(norm(x))`.

In [10]:
class Block(nn.Module):
    def __init__(self, cfg: ModelConfig) -> None:
        super().__init__()
        self.input_layernorm = RMSNorm(cfg.hidden_size, cfg.rms_norm_eps)
        self.self_attn = Attention(cfg)
        self.post_attention_layernorm = RMSNorm(cfg.hidden_size, cfg.rms_norm_eps)
        self.mlp = MLP(cfg)

    def forward(self, x: Tensor, cos: Tensor, sin: Tensor) -> Tensor:
        x = x + self.self_attn(self.input_layernorm(x), cos, sin)
        return x + self.mlp(self.post_attention_layernorm(x))

## 9. The full model
Embedding, a stack of blocks, a final RMSNorm and an output head tied to the embedding. Parameter names
(`model.embed_tokens`, `model.layers.N...`, `lm_head`) match Hugging Face Llama so published weights load
directly. Residual output projections are scaled by `1/sqrt(2*layers)` at init to keep activations bounded.

In [11]:
class LlamaBackbone(nn.Module):
    """Holds embed_tokens / layers / norm so parameter names start with `model.`."""

    def __init__(self, cfg: ModelConfig) -> None:
        super().__init__()
        self.embed_tokens = nn.Embedding(cfg.vocab_size, cfg.hidden_size)
        self.layers = nn.ModuleList(Block(cfg) for _ in range(cfg.num_layers))
        self.norm = RMSNorm(cfg.hidden_size, cfg.rms_norm_eps)


class Transformer(nn.Module):
    def __init__(self, cfg: ModelConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.model = LlamaBackbone(cfg)
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)
        self.apply(self._init_weights)
        for layer in self.model.layers:
            for proj in (layer.self_attn.o_proj, layer.mlp.down_proj):
                proj.weight.data.mul_(1 / math.sqrt(2 * cfg.num_layers))
        if cfg.tie_word_embeddings:
            self.lm_head.weight = self.model.embed_tokens.weight  # one shared matrix

    @staticmethod
    def _init_weights(module: nn.Module) -> None:
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids: Tensor) -> Tensor:
        """input_ids: (B, T) -> logits (B, T, vocab)."""
        positions = torch.arange(input_ids.shape[1], device=input_ids.device)
        cos, sin = rope_cos_sin(positions, self.cfg.head_dim, self.cfg.rope_theta)
        x = self.model.embed_tokens(input_ids)
        for layer in self.model.layers:
            x = layer(x, cos, sin)
        return self.lm_head(self.model.norm(x))

    def num_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters())  # tied matrix counted once

In [12]:
model = Transformer(cfg).to(device=device, dtype=dtype)
print(f"parameters: {model.num_parameters():,}")
print("lm_head tied to embedding:", model.lm_head.weight is model.model.embed_tokens.weight)
[n for n, _ in model.named_parameters()][:12]

parameters: 7,029,888
lm_head tied to embedding: True


['model.embed_tokens.weight',
 'model.layers.0.input_layernorm.weight',
 'model.layers.0.self_attn.q_proj.weight',
 'model.layers.0.self_attn.k_proj.weight',
 'model.layers.0.self_attn.v_proj.weight',
 'model.layers.0.self_attn.o_proj.weight',
 'model.layers.0.post_attention_layernorm.weight',
 'model.layers.0.mlp.gate_proj.weight',
 'model.layers.0.mlp.up_proj.weight',
 'model.layers.0.mlp.down_proj.weight',
 'model.layers.1.input_layernorm.weight',
 'model.layers.1.self_attn.q_proj.weight']

## 10. Shapes through the model
Step through the pieces on the four prompt tokens.

In [13]:
x_ids = torch.tensor([ids], device=device)
vectors = model.model.embed_tokens(x_ids)
print("embedding:", tuple(x_ids.shape), "->", tuple(vectors.shape))

cos, sin = rope_cos_sin(torch.arange(len(ids), device=device), cfg.head_dim, cfg.rope_theta)
layer0 = model.model.layers[0]
print("attention:", tuple(layer0.self_attn(vectors, cos, sin).shape))
print("mlp:      ", tuple(layer0.mlp(vectors).shape))
print("block:    ", tuple(layer0(vectors, cos, sin).shape))

embedding: (1, 4) -> (1, 4, 128)
attention: (1, 4, 128)
mlp:       (1, 4, 128)
block:     (1, 4, 128)


## 11. Full forward pass
One logit per vocabulary entry at every position. The last position predicts the next token.

In [14]:
with torch.no_grad():
    logits = model(x_ids)
print("logits:", tuple(logits.shape), "(B, T, vocab)")
probs = logits[0, -1].float().softmax(-1)
top = probs.topk(5)
for p, i in zip(top.values.tolist(), top.indices.tolist()):
    print(f"{p:.5f}  {tok.id_to_token(i)!r}")
print("uniform guess would be", round(1 / cfg.vocab_size, 7))

logits: (1, 4, 49152) (B, T, vocab)
0.00009  'Ġtime'
0.00006  'Ġcross'
0.00005  'Ġboy'
0.00005  'Ġaviation'
0.00005  'iw'
uniform guess would be 2.03e-05


## 12. Causality check
Changing a future token must not change earlier logits.

In [15]:
a = x_ids.clone()
b = x_ids.clone()
b[0, -1] = (b[0, -1] + 1) % cfg.vocab_size
with torch.no_grad():
    la, lb = model(a), model(b)
print("earlier positions unchanged:", torch.allclose(la[:, :-1], lb[:, :-1], atol=1e-5))
print("last position changed:      ", not torch.allclose(la[:, -1], lb[:, -1], atol=1e-5))

earlier positions unchanged: True
last position changed:       True


## 13. Sampling
Pick the next token from the last position's logits. **Temperature** flattens (high) or sharpens (low) the
distribution, **top-k** keeps only the k most likely tokens, **top-p** keeps the smallest set whose
probability adds up to p. Temperature 0 is greedy.

In [16]:
def sample_next(logits: Tensor, temperature: float, top_k: int | None, top_p: float | None) -> Tensor:
    """logits: (B, vocab) for the last position -> (B, 1) sampled token ids."""
    if temperature <= 0:
        return logits.argmax(dim=-1, keepdim=True)
    logits = logits / temperature
    if top_k:
        kth = torch.topk(logits, min(top_k, logits.size(-1))).values[:, -1:]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    if top_p and top_p < 1.0:
        sorted_logits, idx = torch.sort(logits, descending=True)
        probs = sorted_logits.softmax(dim=-1)
        drop = probs.cumsum(dim=-1) - probs > top_p  # keep the token that crosses the threshold
        sorted_logits = sorted_logits.masked_fill(drop, float("-inf"))
        logits = torch.full_like(logits, float("-inf")).scatter(-1, idx, sorted_logits)
    return torch.multinomial(logits.softmax(dim=-1), num_samples=1)


@torch.no_grad()
def generate(model: Transformer, ids: Tensor, max_new_tokens: int, temperature: float = 1.0,
             top_k: int | None = None, top_p: float | None = None) -> Tensor:
    model.eval()
    for _ in range(max_new_tokens):
        window = ids[:, -model.cfg.max_position_embeddings :]  # crop to the context window
        next_id = sample_next(model(window)[:, -1, :], temperature, top_k, top_p)
        ids = torch.cat((ids, next_id), dim=1)
    return ids

## 14. Generate
Random weights, so expect gibberish. Try `temperature=0` for greedy.

In [17]:
prompt = torch.tensor([tok.encode("Once upon a time").ids], device=device)
out = generate(model, prompt, max_new_tokens=30, temperature=1.0, top_k=50, top_p=0.95)
print(tok.decode(out[0].tolist()))

Once upon a timeHealthy Sacred sorespixel objected""", comparatively）times its reminiseratorChoVitaminVitamin ovary ovaryfindallentary weakenclude its its itsLock wired cpu queer gran marked
